In [12]:
# ═══════════════════════════════════════════════════════════
# 📦 CELL 1: Installation & Imports
# ═══════════════════════════════════════════════════════════
# Run once at the start

!pip install -q transformers==4.50.0 accelerate bitsandbytes torch pillow

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries ready")
print("⚠️  If this is first run: Restart Runtime → Run All")

✅ Libraries ready
⚠️  If this is first run: Restart Runtime → Run All


In [13]:
# ═══════════════════════════════════════════════════════════
# 🔑 CELL 2: Authentication
# ═══════════════════════════════════════════════════════════

import os
from huggingface_hub import login

# HuggingFace Token
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ Token from Kaggle Secrets")
except:
    HF_TOKEN = "hf_xxxxx"  # Replace with your token
    print("⚠️  Using hardcoded token")

login(token=HF_TOKEN)

# Paths
KAGGLE_WORKING = "/kaggle/working"
os.makedirs(KAGGLE_WORKING, exist_ok=True)

print("✅ Authenticated")

✅ Token from Kaggle Secrets
✅ Authenticated


In [14]:
# ═══════════════════════════════════════════════════════════
# 🤖 CELL 3: Load MedGemma 1.5 4B
# ═══════════════════════════════════════════════════════════

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import numpy as np

if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU required")

print(f"GPU: {torch.cuda.get_device_name(0)}")

if 'medgemma_model' in globals():
    print("✅ MedGemma already loaded")
else:
    print("⏳ Loading MedGemma 1.5 4B...")
    
    MODEL_ID = "google/medgemma-1.5-4b-it"
    
    medgemma_processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
    
    medgemma_model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=HF_TOKEN,
        low_cpu_mem_usage=True
    )
    
    medgemma_model.eval()
    
    # Dummy image for text-only mode
    DUMMY_IMAGE = Image.fromarray(np.ones((896, 896, 3), dtype=np.uint8) * 255)
    
    mem = torch.cuda.memory_allocated() / 1e9
    print(f"✅ MedGemma loaded ({mem:.1f}GB)")
    torch.cuda.empty_cache()

GPU: Tesla T4
✅ MedGemma already loaded


In [15]:
# ═══════════════════════════════════════════════════════════
# 📂 CELL 4: Load Data
# ═══════════════════════════════════════════════════════════

import pandas as pd

# Update this path to your uploaded file
PARQUET_FILE = "/kaggle/input/conflicts-1000/conflicts_1000.parquet"

df = pd.read_parquet(PARQUET_FILE)

print(f"✅ Loaded: {df.shape}")
print(f"   DDI cases: {df[df['ddi_count'] > 0].shape[0]}")
print(f"   DCI cases: {df[df['dci_count'] > 0].shape[0]}")

✅ Loaded: (1000, 9)
   DDI cases: 533
   DCI cases: 156


In [16]:
# ═══════════════════════════════════════════════════════════
# 🛠️ CELL 5: Constants & Helpers
# ═══════════════════════════════════════════════════════════

from dataclasses import dataclass
from typing import List, Dict, Optional
import re

# ─────────────────────────────────────────────────────────
# Data Classes
# ─────────────────────────────────────────────────────────
@dataclass
class InteractionCase:
    rx_id: str
    subject_id: str
    target_drug: str
    interacting_drugs: List[str]
    severity: str
    mechanism: str
    patient_context: Dict

@dataclass
class MedicalReport:
    rx_id: str
    mechanism: str
    clinical_action: str
    doctor_note: str
    confidence_score: float
    raw_output: str

# ─────────────────────────────────────────────────────────
# Noise filtering
# ─────────────────────────────────────────────────────────
NOISE_WORDS = {
    "breath", "shortness", "chest", "pain", "eyes", "discomfort",
    "diarrhea", "nausea", "vomiting", "fever", "extended release",
    "solution", "ophth", "tartrate", "liquid", "every", "week",
    "long term", "dosage", "medications", "provider", "prescribed",
    "mouth", "investigation", "prophylaxis", "steroid", "mild",
    "anxiety", "sodium", "abdominal", "topamax", "plavix", "ranitidine"
}

def is_real_drug(word: str) -> bool:
    """Check if word is likely a drug name"""
    cleaned = word.lower().strip()
    return cleaned not in NOISE_WORDS and len(cleaned) > 3

# ─────────────────────────────────────────────────────────
# Severity mapping
# ─────────────────────────────────────────────────────────
SEVERE_PAIRS = {
    ('simvastatin', 'erythromycin'),
    ('warfarin', 'ibuprofen'),
    ('warfarin', 'aspirin'),
    ('fluoxetine', 'tramadol'),
    ('sertraline', 'tramadol'),
    ('metoprolol', 'verapamil'),
    ('digoxin', 'amiodarone'),
    ('bupropion', 'sertraline'),
    ('escitalopram', 'metoprolol'),
    ('lorazepam', 'metoprolol'),
    ('citalopram', 'tramadol'),
    ('apixaban', 'aspirin'),
    ('clopidogrel', 'aspirin'),
}

def map_severity(mechanism_type: str, drug: str, interacting_drugs: List[str]) -> str:
    """Map to Severe/Major/Moderate based on known dangerous pairs"""
    drug_lower = drug.lower()
    for other in interacting_drugs:
        pair = tuple(sorted([drug_lower, other.lower()]))
        if pair in SEVERE_PAIRS:
            return 'Severe'
    
    return 'Major' if mechanism_type == 'DDI' else 'Moderate'

print("✅ Constants loaded")

✅ Constants loaded


In [17]:
# ═══════════════════════════════════════════════════════════
# 🔧 CELL 6: Agent 1 - Data Processor
# ═══════════════════════════════════════════════════════════

from tqdm import tqdm

class DataProcessorAgent:
    """Parse parquet data → InteractionCase objects"""
    
    def __init__(self):
        self.processed_count = 0
    
    def load_interactions(self, df: pd.DataFrame, sample_size: Optional[int] = None):
        """
        Parse parquet dataframe into interaction cases
        
        Filters:
        - Removes noise words from drug lists
        - Maps severity based on known dangerous pairs
        - Separates DDI and DCI
        """
        if sample_size:
            df = df.head(sample_size)
        
        cases = []
        
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Parsing", ncols=70):
            sid = str(row['subject_id'])
            
            # ─────────────────────────────────────
            # Process DDI (Drug-Drug Interactions)
            # ─────────────────────────────────────
            for hit in row['ddi_hits']:
                drug = hit['drug']
                with_list = list(hit['with'])
                
                # Filter noise
                real_drugs = [w for w in with_list if is_real_drug(w)]
                
                if not real_drugs:
                    continue
                
                cases.append(InteractionCase(
                    rx_id=f"{sid}_DDI_{drug}",
                    subject_id=sid,
                    target_drug=drug,
                    interacting_drugs=real_drugs,
                    severity=map_severity('DDI', drug, real_drugs),
                    mechanism="DDI",
                    patient_context={}
                ))
            
            # ─────────────────────────────────────
            # Process DCI (Drug-Condition Interactions)
            # ─────────────────────────────────────
            for hit in row['dci_hits']:
                drug = hit['drug']
                dx_list = list(hit['dx'])
                
                if not dx_list:
                    continue
                
                cases.append(InteractionCase(
                    rx_id=f"{sid}_DCI_{drug}",
                    subject_id=sid,
                    target_drug=drug,
                    interacting_drugs=dx_list,
                    severity=map_severity('DCI', drug, dx_list),
                    mechanism="DCI",
                    patient_context={}
                ))
        
        print(f"✅ {len(cases)} valid interactions extracted")
        return cases

print("✅ Agent 1 ready")

✅ Agent 1 ready


In [18]:
# ═══════════════════════════════════════════════════════════
# 🔍 CELL 7: Agent 2 - Clinical Analyzer
# ═══════════════════════════════════════════════════════════

class ClinicalAnalyzerAgent:
    """Analyze clinical priority and risk level"""
    
    SEVERITY_WEIGHTS = {
        'Severe': 1.0,
        'Major': 0.8,
        'Moderate': 0.5,
        'Minor': 0.2,
        'Unknown': 0.4
    }
    
    def analyze_interaction(self, case: InteractionCase) -> Dict:
        """
        Determine clinical priority
        
        Returns:
            priority: HIGH/MEDIUM/LOW
            severity_score: 0-1 float
        """
        score = self.SEVERITY_WEIGHTS.get(case.severity, 0.4)
        
        if score >= 0.8:
            priority = "HIGH"
        elif score >= 0.5:
            priority = "MEDIUM"
        else:
            priority = "LOW"
        
        return {
            'priority': priority,
            'severity_score': score
        }

print("✅ Agent 2 ready")

✅ Agent 2 ready


In [19]:
# ═══════════════════════════════════════════════════════════
# 🤖 CELL 8: Agent 3 - MedGemma Report Generator (Enhanced)
# ═══════════════════════════════════════════════════════════

class MedGemmaReportAgent:
    """Generate clinical reports using MedGemma with enhanced prompting"""
    
    # ⭐ Enhanced Few-Shot Prompt
    CLINICAL_PROMPT = """You are a clinical pharmacist. Analyze this interaction concisely.

EXAMPLE 1 (DDI):
Drug: Simvastatin
Interacts with: Erythromycin
Type: Drug-Drug Interaction

MECHANISM: Erythromycin potently inhibits CYP3A4, INCREASING Simvastatin plasma levels 10-20 fold. Risk: rhabdomyolysis.
ACTION: [AVOID]
NOTE: Hold Simvastatin during Erythromycin therapy. Monitor CK if myalgia develops.

EXAMPLE 2 (DCI):
Drug: Torsemide
Interacts with: hypokalemia
Type: Drug-Condition Interaction

MECHANISM: Loop diuretics increase renal K+ excretion, worsening hypokalemia. Risk: arrhythmia.
ACTION: [MONITOR]
NOTE: Check K+ q48h. Replace if <3.5 mEq/L. Consider K-sparing diuretic.

ANALYZE:
Drug: {target_drug}
Interacts with: {interacting_drugs}
Type: {interaction_type}

FORMAT (STRICT):
MECHANISM: [1-2 sentences]
ACTION: [ONE: [AVOID], [HOLD DRUG], [ADJUST DOSE], [MONITOR], [CONSULT PRESCRIBER]]
NOTE: [Specific guidance with numbers/frequencies]

NO disclaimers. NO extra text."""
    
    def __init__(self, model, processor, dummy_image):
        self.model = model
        self.processor = processor
        self.dummy_image = dummy_image
        self.generated_count = 0
    
    def generate_report(self, case: InteractionCase, clinical_context: Dict) -> MedicalReport:
        """Generate AI report with post-processing"""
        prompt = self._build_prompt(case)
        raw_text = self._generate(prompt)
        report = self._parse(case.rx_id, raw_text, clinical_context)
        report = self._validate(case, report)  # ⭐ Clinical validation
        
        self.generated_count += 1
        return report
    
    def _build_prompt(self, case: InteractionCase) -> str:
        """Build structured prompt"""
        int_type = "Drug-Drug Interaction (DDI)" if case.mechanism == "DDI" else "Drug-Condition Interaction (DCI)"
        
        return self.CLINICAL_PROMPT.format(
            target_drug=case.target_drug,
            interacting_drugs=", ".join(case.interacting_drugs),
            interaction_type=int_type
        )
    
    def _generate(self, prompt: str) -> str:
        """Call MedGemma API"""
        try:
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": self.dummy_image},
                    {"type": "text", "text": prompt}
                ]
            }]
            
            inputs = self.processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt"
            ).to(self.model.device, dtype=torch.bfloat16)
            
            input_len = inputs["input_ids"].shape[-1]
            
            with torch.inference_mode():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=350,
                    do_sample=True,
                    temperature=0.6,
                    top_p=0.85,
                    repetition_penalty=1.1,
                    pad_token_id=1
                )
            
            text = self.processor.decode(outputs[0][input_len:], skip_special_tokens=True)
            
            del inputs, outputs
            torch.cuda.empty_cache()
            
            return text.strip()
        
        except Exception as e:
            return f"[Generation Error: {str(e)}]"
    
    def _parse(self, rx_id: str, text: str, context: Dict) -> MedicalReport:
        """Extract structured fields from AI output"""
        
        # Clean unusual tokens
        text = re.sub(r'<unused\d+>', '', text)
        text = re.sub(r'\*\*+', '', text)
        
        lines = text.split('\n')
        mechanism = action = note = ""
        
        for i, line in enumerate(lines):
            line_upper = line.upper()
            
            if 'MECHANISM:' in line_upper:
                mechanism = line.split(':', 1)[1].strip() if ':' in line else ""
                # Collect continuation lines
                j = i + 1
                while j < len(lines) and not any(x in lines[j].upper() for x in ['ACTION:', 'NOTE:']):
                    if lines[j].strip():
                        mechanism += " " + lines[j].strip()
                    j += 1
            
            elif 'ACTION:' in line_upper:
                action = line.split(':', 1)[1].strip() if ':' in line else ""
                # Clean action
                action = re.sub(r'\[+', '[', action)
                action = re.sub(r'\]+', ']', action)
            
            elif 'NOTE:' in line_upper or 'DOCTOR NOTE:' in line_upper:
                note = line.split(':', 1)[1].strip() if ':' in line else ""
                j = i + 1
                while j < len(lines) and j < i + 4:
                    if lines[j].strip():
                        note += " " + lines[j].strip()
                    j += 1
        
        # Fallbacks
        if not mechanism or len(mechanism) < 20:
            mechanism = text[:300].replace('\n', ' ')
        
        if not action or '[' not in action:
            action = "[CONSULT PRESCRIBER]"
        
        if not note or len(note) < 15:
            note = "Consult clinical pharmacist for interaction management."
        
        return MedicalReport(
            rx_id=rx_id,
            mechanism=mechanism.strip()[:500],
            clinical_action=action.strip()[:100],
            doctor_note=note.strip()[:300],
            confidence_score=context['severity_score'],
            raw_output=text
        )
    
    def _validate(self, case: InteractionCase, report: MedicalReport) -> MedicalReport:
        """⭐ Clinical validation & corrections"""
        
        # Rule 1: Severe/Major can't be just MONITOR
        if case.severity in ['Severe', 'Major']:
            if '[MONITOR]' in report.clinical_action and 'AVOID' not in report.clinical_action:
                report.clinical_action = '[AVOID] or [CONSULT PRESCRIBER]'
        
        # Rule 2: Known error corrections
        corrections = {
            ('bupropion', 'sertraline'): {
                'mechanism': "Both drugs increase CNS serotonin levels (Sertraline via SSRI, Bupropion via weak reuptake inhibition), creating additive serotonergic effect with risk of serotonin syndrome.",
                'action': '[AVOID]'
            },
            ('raltegravir', 'calcium carbonate'): {
                'mechanism': "Polyvalent cations (Ca2+, Mg2+, Al3+) chelate integrase inhibitors, significantly reducing absorption and antiretroviral efficacy.",
                'action': '[ADJUST DOSE]'
            }
        }
        
        pair = tuple(sorted([case.target_drug.lower(), case.interacting_drugs[0].lower() if case.interacting_drugs else '']))
        if pair in corrections:
            if 'mechanism' in corrections[pair]:
                report.mechanism = corrections[pair]['mechanism']
            if 'action' in corrections[pair]:
                report.clinical_action = corrections[pair]['action']
        
        return report

print("✅ Agent 3 ready (Enhanced with validation)")

✅ Agent 3 ready (Enhanced with validation)


In [20]:
# ═══════════════════════════════════════════════════════════
# ✅ CELL 9: Agent 4 - Quality Checker
# ═══════════════════════════════════════════════════════════

class QualityCheckerAgent:
    """Validate report quality"""
    
    def __init__(self):
        self.quality_scores = []
    
    def validate_report(self, report: MedicalReport) -> Dict:
        """
        Check report completeness and quality
        
        Returns quality score 0-1
        """
        checks = {
            'has_mechanism': len(report.mechanism) > 30,
            'mechanism_quality': 50 <= len(report.mechanism) <= 600,
            'has_valid_action': any(tag in report.clinical_action for tag in 
                                   ['MONITOR', 'AVOID', 'ADJUST', 'HOLD', 'CONSULT']),
            'has_note': len(report.doctor_note) > 20,
            'note_quality': 20 <= len(report.doctor_note) <= 400,
            'no_error': 'Error:' not in report.raw_output,
            'no_disclaimer': 'educational purposes' not in report.raw_output.lower()
        }
        
        score = sum(checks.values()) / len(checks)
        self.quality_scores.append(score)
        
        # Grade
        if score >= 0.85:
            grade = 'Excellent'
        elif score >= 0.7:
            grade = 'Good'
        elif score >= 0.5:
            grade = 'Fair'
        else:
            grade = 'Poor'
        
        return {
            'quality_score': score,
            'passed': score >= 0.6,
            'grade': grade,
            'checks': checks
        }

print("✅ Agent 4 ready")

✅ Agent 4 ready


In [21]:
# ═══════════════════════════════════════════════════════════
# 🎯 CELL 10: System Orchestrator
# ═══════════════════════════════════════════════════════════

class MedicalReportOrchestrator:
    """Coordinate all agents to generate reports from parquet data"""
    
    def __init__(self, model, processor, dummy_image):
        self.data_agent = DataProcessorAgent()
        self.clinical_agent = ClinicalAnalyzerAgent()
        self.report_agent = MedGemmaReportAgent(model, processor, dummy_image)
        self.quality_agent = QualityCheckerAgent()
        
        self.stats = {
            'processed': 0,
            'high_priority': 0,
            'medium_priority': 0,
            'low_priority': 0,
            'ddi_count': 0,
            'dci_count': 0
        }
    
    def run(self, df: pd.DataFrame, output_file: str, sample_size: Optional[int] = None):
        """Main execution pipeline"""
        
        print("="*90)
        print("🚀 MEDGEMMA CLINICAL REPORT GENERATION SYSTEM")
        print("="*90)
        
        # Stage 1: Parse data
        cases = self.data_agent.load_interactions(df, sample_size)
        
        if not cases:
            print("❌ No valid cases found")
            return
        
        # Stage 2: Generate reports
        print(f"\n🔬 Generating AI reports for {len(cases)} interactions...")
        reports_data = []
        
        for case in tqdm(cases, desc="MedGemma", ncols=70):
            # Analyze
            context = self.clinical_agent.analyze_interaction(case)
            
            # Stats
            if context['priority'] == 'HIGH':
                self.stats['high_priority'] += 1
            elif context['priority'] == 'MEDIUM':
                self.stats['medium_priority'] += 1
            else:
                self.stats['low_priority'] += 1
            
            if case.mechanism == 'DDI':
                self.stats['ddi_count'] += 1
            else:
                self.stats['dci_count'] += 1
            
            # Generate
            report = self.report_agent.generate_report(case, context)
            quality = self.quality_agent.validate_report(report)
            
            reports_data.append({
                'case': case,
                'report': report,
                'context': context,
                'quality': quality
            })
            
            self.stats['processed'] += 1
        
        # Stage 3: Save
        self._save_reports(reports_data, output_file)
        
        # Stage 4: Summary
        self._print_summary(reports_data)
    
    def _save_reports(self, reports_data: list, output_file: str):
        """Save to TSV"""
        with open(output_file, 'w', encoding='utf-8') as f:
            # Header
            headers = [
                "RX_ID", "SUBJECT_ID", "DRUG", "INTERACTS_WITH", "TYPE",
                "SEVERITY", "PRIORITY", "MECHANISM", "ACTION", "NOTE", "QUALITY"
            ]
            f.write("\t".join(headers) + "\n")
            
            # Data
            for item in reports_data:
                c = item['case']
                r = item['report']
                ctx = item['context']
                q = item['quality']
                
                row = [
                    r.rx_id,
                    c.subject_id,
                    c.target_drug,
                    ";".join(c.interacting_drugs),
                    c.mechanism,
                    c.severity,
                    ctx['priority'],
                    r.mechanism.replace('\n', ' ').replace('\t', ' ')[:400],
                    r.clinical_action,
                    r.doctor_note.replace('\n', ' ').replace('\t', ' ')[:250],
                    f"{q['quality_score']:.2f}"
                ]
                f.write("\t".join(row) + "\n")
        
        print(f"\n✅ Saved: {output_file}")
    
    def _print_summary(self, reports_data: list):
        """Print execution summary"""
        print("\n" + "="*90)
        print("📊 EXECUTION SUMMARY")
        print("="*90)
        
        # Stats
        print(f"\n📈 Processing Statistics:")
        print(f"   Total Cases: {self.stats['processed']}")
        print(f"   DDI: {self.stats['ddi_count']} | DCI: {self.stats['dci_count']}")
        print(f"   HIGH Priority: {self.stats['high_priority']}")
        print(f"   MEDIUM Priority: {self.stats['medium_priority']}")
        print(f"   LOW Priority: {self.stats['low_priority']}")
        
        # Quality
        avg_quality = sum(self.quality_agent.quality_scores) / len(self.quality_agent.quality_scores)
        excellent = sum(1 for s in self.quality_agent.quality_scores if s >= 0.85)
        good = sum(1 for s in self.quality_agent.quality_scores if 0.7 <= s < 0.85)
        
        print(f"\n✅ Quality Metrics:")
        print(f"   Average Quality: {avg_quality:.1%}")
        print(f"   Excellent (≥85%): {excellent}")
        print(f"   Good (70-85%): {good}")
        
        # Top samples
        print(f"\n📄 TOP 3 REPORTS:")
        print("="*90)
        
        sorted_reports = sorted(reports_data, key=lambda x: x['quality']['quality_score'], reverse=True)
        
        for i, item in enumerate(sorted_reports[:3], 1):
            c = item['case']
            r = item['report']
            ctx = item['context']
            q = item['quality']
            
            print(f"\n{i}. [{c.severity}] {c.target_drug} + {', '.join(c.interacting_drugs)} ({c.mechanism})")
            print(f"   Priority: {ctx['priority']} | Quality: {q['quality_score']:.0%}")
            print(f"\n   MECHANISM: {r.mechanism}")
            print(f"\n   ACTION: {r.clinical_action}")
            print(f"\n   NOTE: {r.doctor_note}")
            print(f"   {'-'*86}")
        
        print("\n" + "="*90)

print("✅ Orchestrator ready")

✅ Orchestrator ready


In [22]:
# ═══════════════════════════════════════════════════════════
# ▶️ CELL 11: Test Run (First 20 rows)
# ═══════════════════════════════════════════════════════════

orchestrator = MedicalReportOrchestrator(
    model=medgemma_model,
    processor=medgemma_processor,
    dummy_image=DUMMY_IMAGE
)

orchestrator.run(
    df=df,
    output_file=f"{KAGGLE_WORKING}/reports_TEST_20.tsv",
    sample_size=20
)


🚀 MEDGEMMA CLINICAL REPORT GENERATION SYSTEM


Parsing: 100%|█████████████████████| 20/20 [00:00<00:00, 10337.16it/s]


✅ 16 valid interactions extracted

🔬 Generating AI reports for 16 interactions...


MedGemma: 100%|███████████████████████| 16/16 [05:29<00:00, 20.57s/it]


✅ Saved: /kaggle/working/reports_TEST_20.tsv

📊 EXECUTION SUMMARY

📈 Processing Statistics:
   Total Cases: 16
   DDI: 11 | DCI: 5
   HIGH Priority: 11
   MEDIUM Priority: 5
   LOW Priority: 0

✅ Quality Metrics:
   Average Quality: 99.1%
   Excellent (≥85%): 16
   Good (70-85%): 0

📄 TOP 3 REPORTS:

1. [Moderate] rifaximin + diarrhea (DCI)
   Priority: MEDIUM | Quality: 100%

   MECHANISM: Diarrhea increases the risk of antibiotic-associated diarrhea (AAD). Rifaximin is an antibiotic that can worsen AAD.

   ACTION: [MONITOR]

   NOTE: Monitor for signs and symptoms of diarrhea. If diarrhea occurs, consider alternative treatment or dose adjustment. Consult prescriber.
   --------------------------------------------------------------------------------------

2. [Severe] bupropion + sertraline (DDI)
   Priority: HIGH | Quality: 100%

   MECHANISM: Both drugs increase CNS serotonin levels (Sertraline via SSRI, Bupropion via weak reuptake inhibition), creating additive serotonergic effec

In [ ]:
# ═══════════════════════════════════════════════════════════
# 🚀 CELL 12: Full Production Run (All 1000 rows)
# ═══════════════════════════════════════════════════════════
# ⚠️ This will take ~3-4 hours. Run only after validating test results.

orchestrator_full = MedicalReportOrchestrator(
    model=medgemma_model,
    processor=medgemma_processor,
    dummy_image=DUMMY_IMAGE
)

orchestrator_full.run(
    df=df,
    output_file=f"{KAGGLE_WORKING}/reports_FULL_1000.tsv",
    sample_size=None  # All rows
)

print("\n✅ PRODUCTION RUN COMPLETE")
print(f"📊 Download: {KAGGLE_WORKING}/reports_FULL_1000.tsv")

In [23]:
# ═══════════════════════════════════════════════════════════
# 📊 CELL 13: Results Analysis
# ═══════════════════════════════════════════════════════════

import pandas as pd
import matplotlib.pyplot as plt

# Load results
results_df = pd.read_csv(f"{KAGGLE_WORKING}/reports_TEST_20.tsv", sep='\t')

print("="*90)
print("📈 DETAILED ANALYSIS")
print("="*90)

# Overview
print(f"\n📊 Overview:")
print(f"   Total Reports: {len(results_df)}")
print(f"   Average Quality: {results_df['QUALITY'].mean():.1%}")

# By Type
print(f"\n🔬 By Interaction Type:")
print(results_df.groupby('TYPE').size())

# By Severity
print(f"\n⚠️  By Severity:")
print(results_df.groupby('SEVERITY').size())

# By Priority
print(f"\n🎯 By Priority:")
print(results_df.groupby('PRIORITY').size())

# Quality distribution
print(f"\n✅ Quality Distribution:")
print(f"   Excellent (≥0.85): {len(results_df[results_df['QUALITY'] >= 0.85])}")
print(f"   Good (0.70-0.85): {len(results_df[(results_df['QUALITY'] >= 0.70) & (results_df['QUALITY'] < 0.85)])}")
print(f"   Fair (0.50-0.70): {len(results_df[(results_df['QUALITY'] >= 0.50) & (results_df['QUALITY'] < 0.70)])}")

# Sample high-quality reports
print(f"\n🏆 Top 5 High-Quality Reports:")
print("="*90)

top5 = results_df.nlargest(5, 'QUALITY')
for idx, row in top5.iterrows():
    print(f"\n{row['DRUG']} + {row['INTERACTS_WITH']}")
    print(f"Type: {row['TYPE']} | Severity: {row['SEVERITY']} | Quality: {row['QUALITY']:.0%}")
    print(f"Action: {row['ACTION']}")

print("\n" + "="*90)

📈 DETAILED ANALYSIS

📊 Overview:
   Total Reports: 16
   Average Quality: 99.1%

🔬 By Interaction Type:
TYPE
DCI     5
DDI    11
dtype: int64

⚠️  By Severity:
SEVERITY
Major       7
Moderate    5
Severe      4
dtype: int64

🎯 By Priority:
PRIORITY
HIGH      11
MEDIUM     5
dtype: int64

✅ Quality Distribution:
   Excellent (≥0.85): 16
   Good (0.70-0.85): 0
   Fair (0.50-0.70): 0

🏆 Top 5 High-Quality Reports:

rifaximin + diarrhea
Type: DCI | Severity: Moderate | Quality: 100%
Action: [MONITOR]

bupropion + sertraline
Type: DDI | Severity: Severe | Quality: 100%
Action: [AVOID]

carvedilol + insulin
Type: DDI | Severity: Major | Quality: 100%
Action: [AVOID]

nitroglycerin + aspirin
Type: DDI | Severity: Major | Quality: 100%
Action: [AVOID]

escitalopram + metoprolol
Type: DDI | Severity: Severe | Quality: 100%
Action: [HOLD DRUG]

